# Classificador 

In [1]:
import pandas as pd

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

In [3]:
import numpy as np

In [4]:
from string import punctuation

In [5]:
from sklearn.feature_extraction.text import CountVectorizer

In [6]:
from sklearn.model_selection import train_test_split

In [7]:
import random

In [8]:
df = pd.read_csv("d:/git/dados/nlp/news_sentiment_analysis.csv", encoding="utf-8")

In [9]:
df = df.drop(columns=["Source", "Author", "URL", "Published At"])

In [10]:
df

,Title,Description,Sentiment,Type
0,Pine View High teacher wins Best in State awar...,"ST. GEORGE — Kaitlyn Larson, a first-year teac...",positive,Business
1,Businesses Face Financial Strain Amid Liquidit...,"Harare, Zimbabwe – Local businesses are grappl...",neutral,Business
2,Musk donates to super pac working to elect Tru...,(marketscreener.com) Billionaire Elon Musk has...,positive,Business
3,US FTC issues warning to franchisors over unfa...,(marketscreener.com) A U.S. trade regulator on...,negative,Business
4,Rooftop solar's dark side,4.5 million households in the U.S. have solar ...,positive,Business
...,...,...,...,...
3495,"Arrow Electronics, Inc. (NYSE:ARW) Shares Purc...",QRG Capital Management Inc. increased its stak...,positive,Technology
3496,"3,120 Shares in NICE Ltd. (NASDAQ:NICE) Bought...",QRG Capital Management Inc. bought a new posit...,positive,Technology
3497,"QRG Capital Management Inc. Has $857,000 Stock...",QRG Capital Management Inc. boosted its stake ...,positive,Technology
3498,Biotechnology Market: Surging Investments and ...,"WESTFORD, Mass., July 18, 2024 /PRNewswire/ --...",neutral,Technology


In [11]:
print(df["Sentiment"].unique())
print(df["Sentiment"].describe())

['positive' 'neutral' 'negative']
count         3500
unique           3
top       positive
freq          2134
Name: Sentiment, dtype: object


In [12]:
# mapping = {'Business': 1, 'Entertainment': 2, 'General': 3, 'Health': 4, 'Science':5, 'Sports':6, 'Technology': 7}
mapping = {
        'positive': 1, 
        'negative': 0,
        'neutral': 0
}
df["sentiment_number"] = df["Sentiment"].map( mapping ) 

In [13]:
df

,Title,Description,Sentiment,Type,sentiment_number
0,Pine View High teacher wins Best in State awar...,"ST. GEORGE — Kaitlyn Larson, a first-year teac...",positive,Business,1
1,Businesses Face Financial Strain Amid Liquidit...,"Harare, Zimbabwe – Local businesses are grappl...",neutral,Business,0
2,Musk donates to super pac working to elect Tru...,(marketscreener.com) Billionaire Elon Musk has...,positive,Business,1
3,US FTC issues warning to franchisors over unfa...,(marketscreener.com) A U.S. trade regulator on...,negative,Business,0
4,Rooftop solar's dark side,4.5 million households in the U.S. have solar ...,positive,Business,1
...,...,...,...,...,...
3495,"Arrow Electronics, Inc. (NYSE:ARW) Shares Purc...",QRG Capital Management Inc. increased its stak...,positive,Technology,1
3496,"3,120 Shares in NICE Ltd. (NASDAQ:NICE) Bought...",QRG Capital Management Inc. bought a new posit...,positive,Technology,1
3497,"QRG Capital Management Inc. Has $857,000 Stock...",QRG Capital Management Inc. boosted its stake ...,positive,Technology,1
3498,Biotechnology Market: Surging Investments and ...,"WESTFORD, Mass., July 18, 2024 /PRNewswire/ --...",neutral,Technology,0


In [14]:
Y = torch.tensor(df["sentiment_number"], dtype=torch.float32)
Y = torch.reshape(Y, (-1, 1))

In [15]:
Y.shape

torch.Size([3500, 1])

In [16]:
table = str.maketrans("", "", punctuation)

# def limpar( texto ):
#     texto_limpo = texto.lower().translate(table)
#     return texto_limpo

def limpar( texto ):
    return texto

In [17]:
df["description_clean"] = df["Description"].apply(limpar)

In [18]:
MAX_PALAVRAS = 500

In [19]:
vetorizador = CountVectorizer(max_features = MAX_PALAVRAS)

In [20]:
texto_vetorizado = vetorizador.fit_transform(df["description_clean"])

In [21]:
dicionario = vetorizador.get_feature_names_out()
dicionario

array(['00', '000', '10', '11', '12', '13', '13f', '14', '15', '16',
       '160', '17', '18', '1st', '20', '2020', '2023', '2024', '20240712',
       '20240716', '20240718', '24', '25', '30', '39', '8211', '8212',
       '8216', '8217', '8220', '8221', '8230', '92nd', 'about',
       'according', 'acquired', 'across', 'action', 'additional', 'after',
       'against', 'agency', 'ai', 'air', 'airman', 'al', 'all', 'also',
       'america', 'american', 'among', 'an', 'analysts', 'and',
       'announced', 'announces', 'annual', 'ap', 'appeared',
       'approximately', 'are', 'army', 'around', 'art', 'as', 'assigned',
       'at', 'attention', 'attorney', 'auf', 'august', 'average', 'back',
       'bank', 'base', 'based', 'be', 'been', 'before', 'bei', 'being',
       'best', 'between', 'biden', 'big', 'board', 'both', 'bought',
       'brown', 'business', 'businesses', 'but', 'by', 'can', 'canada',
       'canadian', 'capabilities', 'capital', 'care', 'center', 'channel',
       'che',

In [22]:
X = torch.tensor(texto_vetorizado.toarray(), dtype=torch.float32)
X

tensor([[0., 0., 1.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 1.,  ..., 0., 0., 0.]])

In [23]:
print("X: ", X.dtype, X.shape, X.ndim)
print("Y: ", Y.dtype, Y.shape, Y.ndim)

X:  torch.float32 torch.Size([3500, 500]) 2
Y:  torch.float32 torch.Size([3500, 1]) 2


In [24]:
X_treino, X_teste, Y_treino, Y_teste = train_test_split(X, Y, test_size=0.2, random_state=100)

In [25]:
len(X_treino)
# X_treino[0].sum()

2800

In [30]:
# modelo = nn.Linear(in_features = MAX_PALAVRAS, out_features=1modelo = nn.Sequential(
modelo = nn.Sequential(
    nn.Linear(in_features = MAX_PALAVRAS, out_features=1),
    nn.Sigmoid()
)


In [31]:
criterio = nn.BCELoss() # Binary Cross Entropy
otimizador = optim.SGD( modelo.parameters(), lr=0.01 )

In [33]:
for epoca in range(1, 3000):
    Y_hat = modelo( X_treino )
    loss = criterio( Y_hat, Y_treino )
    otimizador.zero_grad()
    loss.backward()
    otimizador.step()
    if epoca % 100 == 0:
        print(f"Epoca: {epoca}\tLoss:{loss}")

Epoca: 100	Loss:0.49370139837265015
Epoca: 200	Loss:0.48943838477134705
Epoca: 300	Loss:0.48548948764801025
Epoca: 400	Loss:0.4818103015422821
Epoca: 500	Loss:0.4783651828765869
Epoca: 600	Loss:0.4751252830028534
Epoca: 700	Loss:0.4720667898654938
Epoca: 800	Loss:0.4691700041294098
Epoca: 900	Loss:0.46641820669174194
Epoca: 1000	Loss:0.4637972116470337
Epoca: 1100	Loss:0.46129491925239563
Epoca: 1200	Loss:0.4589007496833801
Epoca: 1300	Loss:0.4566057622432709
Epoca: 1400	Loss:0.45440182089805603
Epoca: 1500	Loss:0.45228174328804016
Epoca: 1600	Loss:0.4502395689487457
Epoca: 1700	Loss:0.44826948642730713
Epoca: 1800	Loss:0.44636666774749756
Epoca: 1900	Loss:0.44452646374702454
Epoca: 2000	Loss:0.44274500012397766
Epoca: 2100	Loss:0.4410184919834137
Epoca: 2200	Loss:0.43934378027915955
Epoca: 2300	Loss:0.4377177357673645
Epoca: 2400	Loss:0.4361375570297241
Epoca: 2500	Loss:0.434600830078125
Epoca: 2600	Loss:0.43310511112213135
Epoca: 2700	Loss:0.4316483438014984
Epoca: 2800	Loss:0.430228

In [34]:
vetorizador_predict = CountVectorizer(max_features = MAX_PALAVRAS, vocabulary=dicionario)

In [35]:
indice = random.randint(0, 3500)
predict_set = [ df["description_clean"][indice] ]
tipo = [ df["Sentiment"][indice] ]
print(f"Indice: {indice}\t\tTipo: {tipo}")
print(predict_set)

# predict_set = [
#     # "the fruitwatch initiative a groundbreaking citizen science project has significantly enhanced the accuracy of predicting flowering times for fruit trees across great britain this improvement is vital for the agricultural sector enabling better planning for pest management and pollinator support which are crucial for maintaining optimal fruit yield and quality"
#     # "researchers at the institute for systems biology in seattle found that bowel movement frequency could predict kidney and liver damage as well as mental health issues like depression"
#     # "ap entertainment writer new york ap — richard simmons television8217s hyperactive court jester of physical fitness who built a miniempire in his trademark tank tops and short shorts by urging the overweight to exercise and eat better died saturday he turned 76 on friday los angeles police and fire departments say they responded to athe post richard simmons a fitness guru who mixed laughs and sweat dies at 76 appeared first on kvia"
# ]

Indice: 587		Tipo: ['positive']
['The Mahomes family is growing once again. Kansas City Chiefs quarterback Patrick Mahomes and his wife Brittany announced on Friday that they are expecting their third child. The couple shared a video and some images from a photo shoot with their two other children and captioned it, &#8220;Round three, here we come.&#8221; Congrats to the...The post Patrick Mahomes, wife Brittany announce big personal news appeared first on Larry Brown Sports.']


In [44]:
X_predict = vetorizador_predict.fit_transform( predict_set )
list_reverse = ['Negative', 'Positive']
with torch.no_grad():
    X_pred = torch.tensor(X_predict.toarray(), dtype=torch.float32)
    resposta = modelo( X_pred )
    indice = round( resposta.item() )
    print(f"Este texto é sobre {resposta} {indice} {list_reverse[indice]}")
# X_predict.toarray()

Este texto é sobre tensor([[0.8819]]) 1 Pòsitive


In [ ]:
X_pred

In [55]:
modelo.eval()

with torch.no_grad():
    vrespostas = modelo( X_teste )
    Y_teste_hat = torch.tensor([round( resp.item() ) for resp in vrespostas], dtype=torch.float32).reshape(-1, 1)

In [57]:
resultados

tensor([[ 1.],
        [ 0.],
        [-1.],
        [ 1.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.],
        [-1.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 1.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.],
        [-1.],
        [ 0.],
        [-1.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.],
        [-1.],
        [ 0.],
        [ 0.],
        [ 1.],
        [ 1.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 1.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.],
        [-1.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 

In [63]:
tp = 0
tn = 0
fp = 0
fn = 0
for idx, resultado_hat in enumerate(Y_teste_hat):
    resultado = Y_teste[idx]
    if resultado_hat == 1 and resultado == 1:
        tp += 1
    elif resultado_hat == 0 and resultado == 1:
        fn += 1
    elif resultado_hat == 1 and resultado == 0:
        fp += 1
    elif resultado_hat == 0 and resultado == 0:
        tn += 1

print(f"Total testado: {len(Y_teste_hat)}")
print(f"TP: {tp}\tTN: {tn}\tFP: {fp}\tFN: {fn}")

resultados = (Y_teste_hat - Y_teste)
total = len(resultados)
total_erros = len(resultados[ resultados > 0 ])
total_acertos = total - total_erros

acerto = total_acertos / total
print("Acerto do modelo ==>", acerto)
precision = tp / (tp + fp)
recall = tp / (tp + fn)
acuracy = (tp + tn) / (tp + tn + fp + fn)
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"Acuracy: {acuracy}")


Total testado: 700
TP: 361	TN: 197	FP: 78	FN: 64
Acerto do modelo ==> 0.8885714285714286
Precision: 0.8223234624145785
Recall: 0.8494117647058823
Acuracy: 0.7971428571428572


Com 500 palavras o Acerto do modelo e texto limpo ==> 0.8957142857142857
Com 500 palavras o Acerto do modelo e texto sem limpar ==>  0.9128571428571428
Com 1000 palavras o Acerto do modelo e texto limpo  ==> 0.9057142857142857
Com 2000 palavras o Acerto do modelo e texto limpo  ==> 0.8971428571428571

In [ ]:
# mapping = {
#         'Business': [0, 0, 0, 0, 0, 0, 1], 
#         'Entertainment': [0, 0, 0, 0, 0, 1, 0],
#         'General': [0, 0, 0, 0, 1, 0, 0],
#         'Health': [0, 0, 0, 1, 0, 0, 0], 
#         'Science': [0, 0, 1, 0, 0, 0, 0],
#         'Sports': [0, 1, 0, 0, 0, 0, 0], 
#         'Technology': [1, 0, 0, 0, 0, 0, 0]
# }
# list_reverse = ['Technology', 'Sports', 'Science', 'Health', 'General', 'Entertainment', 'Business']